# Combine methods figure components

Recreates the full methods figure by compositing the pre-rendered components in `figures/<VERSION>/` (produced by `create_methods_figure_components.ipynb` with the matching config) onto a single canvas.

Everything is placed in a fixed 2434 × 1053 pixel coordinate system (x right, y down) on one matplotlib axes, so positions below are directly in output-image pixels. The result is saved to `figures/<VERSION>/methods_figure.png`.

Version-dependent pieces: `N_YEARS` sets how many sheets are drawn in the two annual-product globe stacks and the "*N*-year composites" panel title (v9 = WY2015–2024 = 10, v10 = WY2015–2025 = 11). Note the committed v9 render (`figures/v9/methods_figure.png`, the manuscript Fig. 1) drew 11 sheets — faithfully measured off the original figure — so a v9 rerun with `N_YEARS = 10` will differ slightly from it.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.patches import (Rectangle, FancyArrow, FancyArrowPatch,
                                FancyBboxPatch)
from matplotlib import font_manager
from PIL import Image

# dataset version this render corresponds to: components are read from
# figures/<VERSION>/ and the combined figure is written there too. N_YEARS
# drives the annual-stack sheet counts and the "<N>-year composites" label
# (v9/WY2015-2024 -> 10, v10/WY2015-2025 -> 11).
VERSION = 'v10'
N_YEARS = 11

FIGDIR = f'figures/{VERSION}'
OUT = f'{FIGDIR}/methods_figure.png'

# the original figure uses Oswald (fall back to the default font if absent)
VAR_FONT = '/usr/local/share/fonts/Oswald-VariableFont_wght.ttf'
if os.path.exists(VAR_FONT):
    font_manager.fontManager.addfont(VAR_FONT)
    plt.rcParams['font.family'] = 'Oswald'

C = dict(
    teal_fill='#b3e0e5', teal_edge='#0096a6', teal_text='#0096a6',
    yellow_fill='#fff1d2', yellow_edge='#fbd067', yellow_text='#e8c33c',
    green_fill='#cae9da', green_edge='#48a578', green_text='#3a9a62',
    pink_fill='#e6cad8', pink_edge='#a64b78',
    pinkarrow_fill='#f5d9e2', pinkarrow_edge='#d989a8', pinkarrow_text='#d6558d',
    label_fill='#ececec', label_edge='#1a1a1a',
    red='#ff0000', green='#00b050', blue='#0070ff',
    purple='#7030a0', darkred='#a00000',
    greenarrow_fill='#cde9cf', greenarrow_edge='#69b06f', greenarrow_text='#3aa14a',
)

IMG = {}
def img(name):
    if name not in IMG:
        IMG[name] = np.asarray(Image.open(f'{FIGDIR}/{name}.png'))
    return IMG[name]

## Drawing helpers

Small primitives used throughout: image placement (with optional border and crop), rounded label boxes, block/curved arrows, pixel markers, isometric "pancake" image stacks (45°-rotated, vertically squashed sheets piled downward, with pixel markers placeable at any image-relative point), tilted globe pancake stacks, and a horizontal curly brace.

In [ ]:
def add_img(ax, name, x0, y0, w, h, zorder=5, border=None, blw=1.5, crop=None):
    a = img(name)
    if crop:  # (u0, v0, u1, v1) as fractions of the source width/height
        H, W = a.shape[:2]
        u0, v0, u1, v1 = crop
        a = a[int(v0 * H):int(v1 * H), int(u0 * W):int(u1 * W)]
    im = ax.imshow(a, extent=(x0, x0 + w, y0 + h, y0), zorder=zorder,
                   interpolation='bilinear', aspect='auto')
    if border:
        ax.add_patch(Rectangle((x0, y0), w, h, fill=False, ec=border, lw=blw,
                               zorder=zorder + 0.004))
    return im

def box_label(ax, cx, cy, w, h, text, fs, r=None, zorder=20, fc=None, ec=None,
              tc='black', lw=2.8, weight='normal'):
    """Fixed-size rounded label box with centered text (the target's label
    boxes are wider than the text, so they can't be drawn as text bboxes)."""
    r = r if r is not None else 0.3 * h
    ax.add_patch(FancyBboxPatch((cx - w / 2, cy - h / 2), w, h,
                                boxstyle=f'round,pad=0,rounding_size={r}',
                                mutation_scale=1, fc=fc or C['label_fill'],
                                ec=ec or C['label_edge'], lw=lw, zorder=zorder))
    ax.text(cx, cy, text, ha='center', va='center', fontsize=fs, color=tc,
            weight=weight, zorder=zorder + 0.01)

# every Rainier chip gets its pixel marker at the same relative position
PIXEL_UV = (0.708, 0.642)

def block_arrow(ax, x0, y0, x1, y1, body, head_w, head_l, fc, ec, lw=2.5, zorder=8):
    ax.add_patch(FancyArrow(x0, y0, x1 - x0, y1 - y0, width=body,
                            head_width=head_w, head_length=head_l,
                            length_includes_head=True, fc=fc, ec=ec, lw=lw,
                            zorder=zorder, joinstyle='miter'))

def curved_arrow(ax, x0, y0, x1, y1, color, rad=0.2, lw=2.5, ls='solid',
                 zorder=15, mut=22):
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), connectionstyle=f'arc3,rad={rad}',
                                 arrowstyle='-|>', mutation_scale=mut, lw=lw,
                                 color=color, linestyle=ls, zorder=zorder))

def bezier_arrow(ax, p0, c1, c2, p1, color, lw=2.5, ls='solid', zorder=15,
                 mut=22):
    """Cubic-bezier arrow, for the S-shaped hugs a plain arc3 can't do."""
    from matplotlib.path import Path
    path = Path([p0, c1, c2, p1], [Path.MOVETO, Path.CURVE4, Path.CURVE4,
                                   Path.CURVE4])
    ax.add_patch(FancyArrowPatch(path=path, arrowstyle='-|>',
                                 mutation_scale=mut, lw=lw, color=color,
                                 linestyle=ls, zorder=zorder))

def pixel_square(ax, x, y, size=11, zorder=18, lw=2.0):
    ax.add_patch(Rectangle((x - size / 2, y - size / 2), size, size, fill=False,
                           ec='black', lw=lw, zorder=zorder))

def iso_transform(cx, cy, squash=0.55):
    return mtransforms.Affine2D().rotate_deg(45).scale(1, squash).translate(cx, cy)

def iso_point(cx, cy, w, u, v, squash=0.55):
    """Data coords of the image-relative point (u right, v down in [0,1]) on an
    iso sheet centered at (cx, cy)."""
    s = w / np.sqrt(2)
    return iso_transform(cx, cy, squash).transform((-s / 2 + u * s,
                                                    -s / 2 + v * s))

def iso_marker(ax, cx, cy, w, u, v, color, squash=0.55, zorder=8, d=None, lw=2.4):
    """Open square at (u, v) on the top sheet, transformed with the sheet so it
    reads as the same map pixel."""
    s = w / np.sqrt(2)
    d = d or 0.075 * s
    r = Rectangle((-s / 2 + u * s - d / 2, -s / 2 + v * s - d / 2), d, d,
                  fill=False, ec=color, lw=lw, zorder=zorder, joinstyle='miter')
    r.set_transform(iso_transform(cx, cy, squash) + ax.transData)
    ax.add_patch(r)

def iso_sheet(ax, name, cx, cy, w, squash=0.55, zorder=5, border=None, blw=2.2):
    """One image 'lying flat': rotated 45° then squashed vertically, so the
    square becomes a diamond of full width w (height w*squash), centered at
    (cx, cy)."""
    s = w / np.sqrt(2)
    t = iso_transform(cx, cy, squash)
    im = ax.imshow(img(name), extent=(-s / 2, s / 2, s / 2, -s / 2),
                   zorder=zorder, interpolation='bilinear', aspect='auto')
    im.set_transform(t + ax.transData)
    if border:
        r = Rectangle((-s / 2, -s / 2), s, s, fill=False, ec=border, lw=blw,
                      zorder=zorder + 0.004, joinstyle='miter')
        r.set_transform(t + ax.transData)
        ax.add_patch(r)

def iso_stack(ax, name, cx, cy, w, n, dy, border, squash=0.55, blw=2.2,
              zorder=5, guides=False):
    """Isometric pancake pile: top sheet centered at (cx, cy), the others
    directly below at dy spacing (only their lower edges peek out)."""
    for i in range(n - 1, -1, -1):
        iso_sheet(ax, name, cx, cy + i * dy, w, squash=squash,
                  zorder=zorder + (n - i) * 0.01, border=border, blw=blw)
    if guides:  # faint dotted verticals dropping from the top sheet's corners
        # each guide ends at the matching corner of the LAST sheet: side
        # guides stop at its side-corner level, the front guide at its front
        # (bottom) corner
        h = w * squash
        drop = (n - 1) * dy
        for px, py in [(cx - w / 2, cy), (cx + w / 2, cy),
                       (cx, cy + h / 2)]:
            ax.plot([px, px], [py, py + drop], color=border, lw=1.4,
                    ls=(0, (2, 3)), alpha=0.75, zorder=zorder + n * 0.01 + 0.05)

def globe_stack(ax, name, cx, cy, w, n, dy, rot=-12, squash=0.85, zorder=5):
    """Pancake pile of globe maps: every sheet is the (slightly squashed) map
    tilted by rot, piled straight down at dy spacing. Antarctica stays visible
    as the white slivers on each exposed rim, like the target; a thin dark
    strip along each sheet's bottom edge separates the pancakes."""
    a = np.array(img(name), copy=True)
    a = a[: int(a.shape[0] * 0.97)]  # drop the antialiased bottom edge
    if a.shape[-1] == 4:
        solid = a[-3:, :, 3] > 128
        a[-3:, :, :3][solid] = (60, 60, 60)
    h = w * a.shape[0] / a.shape[1]
    for i in range(n - 1, -1, -1):
        t = (mtransforms.Affine2D().scale(1, squash).rotate_deg(rot)
             .translate(cx, cy + i * dy))
        im = ax.imshow(a, extent=(-w / 2, w / 2, h / 2, -h / 2),
                       zorder=zorder + (n - i) * 0.01,
                       interpolation='bilinear', aspect='auto')
        im.set_transform(t + ax.transData)

def draw_brace(ax, x0, x1, y, depth=26, color='black', lw=3, zorder=15):
    """Horizontal under-brace: ends at y, center tip at y+depth (points down)."""
    n = 501
    x = np.linspace(x0, x1, n)
    beta = 150.0 / (x1 - x0)
    xh = x[: n // 2 + 1]
    yh = (1 / (1 + np.exp(-beta * (xh - xh[0])))
          + 1 / (1 + np.exp(-beta * (xh - xh[-1]))))
    yy = np.concatenate((yh, yh[-2::-1]))
    yy = y + (yy - 0.5) * depth
    ax.plot(x, yy, color=color, lw=lw, zorder=zorder, solid_capstyle='round')

## Background panels and corner titles

The three colored panels: **runoff onset identification** (teal), **global dataset** (yellow), and the nested **N-year composites** (green, titled from `N_YEARS`).

In [ ]:
def draw_panels(ax):
    ax.add_patch(Rectangle((11, 11), 1419, 1031, fc=C['teal_fill'],
                           ec=C['teal_edge'], lw=9.4, zorder=0))
    ax.add_patch(Rectangle((1445, 11), 977, 1031, fc=C['yellow_fill'],
                           ec=C['yellow_edge'], lw=9.4, zorder=0))
    # the green composites outline stops short of the yellow panel outline —
    # there is a small gap of yellow fill between the two on every side
    ax.add_patch(Rectangle((1815, 38), 581, 916, fc=C['green_fill'],
                           ec=C['green_edge'], lw=10, zorder=1))
    ax.text(1422, 1008, 'runoff onset identification', ha='right', va='center',
            fontsize=42, color=C['teal_text'], weight='bold', zorder=30)
    ax.text(2406, 1004, 'global dataset', ha='right', va='center', fontsize=36,
            color=C['yellow_text'], weight='bold', zorder=30)
    ax.text(2384, 919, f'{N_YEARS}-year composites', ha='right', va='center',
            fontsize=35, color=C['green_text'], weight='bold', zorder=30)

## Left panel — runoff onset identification

Snow phenology inputs (Mt. Rainier example maps), Sentinel-1 RTC relative-orbit stacks with the insufficient-revisit filter, and the "for each pixel" arrow.

In [ ]:
# snow phenology chip geometry (shared with draw_left_connections)
SP_XS = [38, 249, 460]
SP_Y, SP_W, SP_H = 104, 205, 203

def draw_snow_phenology(ax):
    box_label(ax, 351, 50, 470, 48, 'Snow phenology dataset', fs=24, r=17)
    names = ['rainier_max_consec_snow_days_2023', 'rainier_SAD_dowy_2023',
             'rainier_SDD_dowy_2023']
    subs = ['consec. snow days', 'appearance date', 'disappearance date']
    for x, name, sub in zip(SP_XS, names, subs):
        ax.text(x + SP_W / 2, 90, sub, ha='center', va='center', fontsize=19,
                weight='bold', zorder=20)
        add_img(ax, name, x, SP_Y, SP_W, SP_H, border='black', blw=3)
        pixel_square(ax, x + PIXEL_UV[0] * SP_W, SP_Y + PIXEL_UV[1] * SP_H,
                     size=13, lw=2.5)

# backscatter curve definitions shared by the plot and the dashed S1 arrows
XX = np.linspace(134, 1065, 900)
CURVES = dict(
    blue=(598, 522, 11, 0.8, 703, 38, 129),
    red=(554, 500, 13, 2.6, 745, 30, 175),
    green=(660, 555, 13, 4.6, 789, 35, 110),
)

FREQ = dict(red=(27, 57), blue=(58, 110), green=(38, 80))

def curve_y(col, x=None):
    base, post, amp, ph, dipc, dipw, depth = CURVES[col]
    f1, f2 = FREQ[col]
    xx = XX if x is None else np.asarray(x, float)
    pre = base + amp * np.sin((xx - 134) / f1 + ph) \
               + 0.4 * amp * np.sin((xx - 134) / f2 + 2.1 * ph)
    po = post + 0.5 * amp * np.sin((xx - 134) / 80 + ph)
    s = 1 / (1 + np.exp(-(xx - dipc - 50) / 28))
    y = pre * (1 - s) + po * s
    y += depth * np.exp(-(((xx - dipc) / dipw) ** 2))
    return y

def draw_s1_stacks(ax):
    box_label(ax, 1093, 53, 572, 44,
              'Sentinel-1 RTC grouped by relative orbit', fs=22, r=16)
    # input cluster: one mini pile per relative orbit, incl. the purple orbit
    # that the revisit filter will remove
    iso_stack(ax, 's1_rtc_relorbit_64', 893, 107, 88, n=6, dy=7,
              border=C['green'], blw=1.6, zorder=5.0)
    iso_stack(ax, 's1_rtc_relorbit_13', 841, 119, 88, n=10, dy=8,
              border=C['red'], blw=1.6, zorder=5.2)
    iso_stack(ax, 's1_rtc_relorbit_115', 909, 150, 88, n=6, dy=7,
              border=C['blue'], blw=1.6, zorder=5.4)
    iso_stack(ax, 's1_rtc_relorbit_137', 867, 195, 88, n=4, dy=7,
              border=C['purple'], blw=1.6, zorder=5.6)
    # filtered piles: green (back), red, blue (front) — taller than the input
    # minis; sizes/positions measured off the target
    piles = dict(
        green=('s1_rtc_relorbit_64', 1182, 143, 180, 7, 16.5),
        red=('s1_rtc_relorbit_13', 1068, 148, 180, 10, 17.3),
        blue=('s1_rtc_relorbit_115', 1284, 248, 180, 6, 17.5),
    )
    for col, z in [('green', 5.0), ('red', 5.3), ('blue', 5.6)]:
        name, cx, cy, w, n, dy = piles[col]
        iso_stack(ax, name, cx, cy, w, n=n, dy=dy, border=C[col], blw=2.2,
                  zorder=z, guides=True, squash=0.53)
    # white block arrow: remove orbits with insufficient revisit — drawn IN
    # FRONT of the piles, its head overlapping the red stack
    block_arrow(ax, 823, 256, 1060, 256, body=68, head_w=102, head_l=65,
                fc='white', ec='black', lw=3, zorder=6.5)
    ax.text(910, 255, 'remove orbits with\ninsufficient revisit', ha='center',
            va='center', fontsize=17.5, zorder=6.6, linespacing=1.4)
    # pixel marker on each pile's top sheet (same image-relative pixel as the
    # Rainier chips) + dashed S-curve that hugs the pile's right side, then
    # drops onto the same-colored backscatter curve pointing straight down
    hugs = dict(red=((100, 140), 55, 918), green=((50, 120), 40, 945),
                blue=((85, 100), 50, 1012))
    for col in ('red', 'green', 'blue'):
        name, cx, cy, w, n, dy = piles[col]
        mx, my = iso_point(cx, cy, w, *PIXEL_UV, squash=0.53)
        iso_marker(ax, cx, cy, w, *PIXEL_UV, C[col], zorder=8.2, squash=0.53)
        (dx1, dy1), dx2, ex = hugs[col]
        ey = float(curve_y(col, ex)) - 5
        bezier_arrow(ax, (mx, my + 6), (mx + dx1, my + dy1),
                     (ex + dx2, ey - 140), (ex, ey), C[col],
                     ls=(0, (4.5, 3)), lw=3.5, zorder=14)

def draw_pink_arrow(ax):
    block_arrow(ax, 150, 288, 150, 478, body=192, head_w=264, head_l=78,
                fc=C['pinkarrow_fill'], ec=C['pinkarrow_edge'], lw=2.5,
                zorder=4.5)
    ax.text(150, 372, 'for each pixel\nwith ≥56 consec.\nsnow days',
            ha='center', va='center', fontsize=22, weight='bold',
            color=C['pinkarrow_text'], zorder=9, linespacing=1.15)

The schematic backscatter-vs-DOWY plot: snow appearance/disappearance bounds, search window, per-orbit backscatter minima, and the median-of-orbits runoff onset. Then the tile-level products and the arrows connecting everything.

In [ ]:
def draw_backscatter_plot(ax):
    # pink box
    ax.add_patch(Rectangle((43, 436), 1060, 531, fc=C['pink_fill'],
                           ec=C['pink_edge'], lw=8, zorder=2))
    # shaded bands (drawn above pink fill)
    ax.add_patch(Rectangle((341, 478), 245, 325, fc='gray', alpha=0.45,
                           ec='none', zorder=3))
    ax.add_patch(Rectangle((586, 478), 338, 325, fc='white', ec='none', zorder=3))
    # axes
    ax.plot([116, 116], [468, 806], color='black', lw=5.5, zorder=10,
            solid_capstyle='projecting')
    ax.plot([116, 1077], [806, 806], color='black', lw=5.5, zorder=10,
            solid_capstyle='projecting')
    ax.text(97, 614, 'backscatter', rotation=90, ha='center', va='center',
            fontsize=29, weight='bold', zorder=10)
    ax.text(412, 851, 'DOWY', ha='center', va='center', fontsize=30,
            weight='bold', zorder=10)
    ax.text(124, 826, 'WY start', ha='center', va='center', fontsize=27, zorder=10)
    ax.text(1040, 826, 'WY end', ha='center', va='center', fontsize=26, zorder=10)
    # vertical dashed lines
    ax.plot([341, 341], [478, 803], color=C['purple'], lw=4.5, ls=(0, (6, 5)),
            zorder=11)
    ax.plot([846, 846], [478, 803], color=C['darkred'], lw=4.5, ls=(0, (6, 5)),
            zorder=11)
    # +16 days annotation
    ax.annotate('', xy=(925, 633), xytext=(848, 633),
                arrowprops=dict(arrowstyle='<|-|>', color='black', lw=4,
                                mutation_scale=20), zorder=12)
    ax.text(885, 694, '+16\ndays', ha='center', va='center', fontsize=28,
            weight='bold', zorder=12, linespacing=1.07)
    # backscatter curves
    for col in ('blue', 'red', 'green'):
        yy = curve_y(col)
        ax.plot(XX, yy, color=C[col], lw=4, zorder=13, solid_capstyle='round')
        dipc = CURVES[col][4]
        ymin = yy[np.argmin(np.abs(XX - dipc))]
        ax.plot([dipc, dipc], [ymin + 6, 803], color=C[col], lw=3,
                ls=(0, (2, 3)), zorder=12)
    # brace + median text
    draw_brace(ax, 692, 796, 812, depth=36)
    ax.text(799, 861, 'median(', ha='right', va='center', fontsize=26,
            weight='bold', zorder=15)
    ax.text(799, 861, 'DOWY_orbit1,', ha='left', va='center', fontsize=20,
            weight='bold', color=C['red'], zorder=15)
    ax.text(799, 898, 'DOWY_orbit2,', ha='left', va='center', fontsize=20,
            weight='bold', color=C['green'], zorder=15)
    # closing paren stays black: draw the full string in black, then overdraw
    # the shared prefix in blue exactly on top (same anchor -> same glyphs)
    ax.text(799, 935, 'DOWY_orbit3)', ha='left', va='center', fontsize=20,
            weight='bold', color='black', zorder=15)
    ax.text(799, 935, 'DOWY_orbit3', ha='left', va='center', fontsize=20,
            weight='bold', color=C['blue'], zorder=15.01)

def draw_left_connections(ax):
    # pixel marker on appearance date map -> purple line, disappearance ->
    # dark red line
    px, py = PIXEL_UV[0] * SP_W, SP_Y + PIXEL_UV[1] * SP_H
    curved_arrow(ax, SP_XS[1] + px, py + 8, 344, 464, 'black', rad=0.05, lw=2.5)
    # disappearance date -> dark-red dashed line: bezier so the head arrives
    # pointing straight down onto the line
    x0, y0 = SP_XS[2] + px, py + 8
    bezier_arrow(ax, (x0, y0), (x0 + 160, y0 + 40), (845, 350), (845, 464),
                 'black', lw=2.5)

def draw_tile_products(ax):
    # two identical square tiles; wide stadium label boxes overlap each map's
    # upper area and stretch right across the teal border
    add_img(ax, 'rainier_runoff_onset_2023', 1136, 451, 226, 226,
            border='black', blw=4, zorder=6)
    box_label(ax, 1250, 503, 275, 57, 'tile runoff onset', fs=22, r=28, lw=3)
    pixel_square(ax, 1136 + PIXEL_UV[0] * 226, 451 + PIXEL_UV[1] * 226,
                 size=14, lw=2.6)
    add_img(ax, 'rainier_temporal_resolution_2023', 1136, 700, 226, 226,
            border='black', blw=4, zorder=6)
    box_label(ax, 1250, 735, 275, 57, 'tile temporal res.', fs=22, r=28, lw=3)
    pixel_square(ax, 1136 + PIXEL_UV[0] * 226, 700 + PIXEL_UV[1] * 226,
                 size=14, lw=2.6)
    # median() -> tile runoff onset pixel: up past the LEFT side of 'WY end',
    # then across (just under the curve ends, perpendicular over the pink
    # border) to the pixel, arriving horizontally
    from matplotlib.path import Path
    path = Path([(945, 935), (1005, 938), (985, 880), (983, 800),
                 (981, 690), (1030, 599), (1287, 597)],
                [Path.MOVETO] + [Path.CURVE4] * 6)
    ax.add_patch(FancyArrowPatch(path=path, arrowstyle='-|>',
                                 mutation_scale=22, lw=2.5, color='black',
                                 zorder=15))
    # white arrows to the annual stacks
    block_arrow(ax, 1388, 592, 1538, 452, body=34, head_w=80, head_l=55,
                fc='white', ec='black', lw=2.5, zorder=8)
    block_arrow(ax, 1375, 851, 1462, 851, body=24, head_w=64, head_l=46,
                fc='white', ec='black', lw=2.5, zorder=8)

## Middle and right panels — global dataset

Annual product stacks (tilted globe pancake piles, one sheet per water year), the composite-calculation arrow, and the three N-year composite maps.

In [ ]:
def draw_annual_stacks(ax):
    # one sheet per water year in the record (N_YEARS)
    globe_stack(ax, 'global_runoff_onset_2020', 1611, 258, 336, n=N_YEARS,
                dy=15.7, rot=-25, squash=0.79, zorder=5)
    box_label(ax, 1626, 334, 320, 56, 'annual runoff onset', fs=23.5, r=22,
              lw=3)
    globe_stack(ax, 'global_temporal_resolution_2020', 1611, 779, 336,
                n=N_YEARS, dy=15.7, rot=-25, squash=0.79, zorder=5)
    box_label(ax, 1626, 852, 320, 56, 'annual temporal res.', fs=23.5, r=22,
              lw=3)
    # green composite arrow
    block_arrow(ax, 1493, 586, 1810, 586, body=112, head_w=172, head_l=70,
                fc=C['greenarrow_fill'], ec=C['greenarrow_edge'], lw=2.5, zorder=7)
    ax.text(1502, 584, 'calculate composite products\nfrom annual products',
            ha='left', va='center', fontsize=19, weight='bold',
            color=C['greenarrow_text'], zorder=8, linespacing=1.3)


def draw_composites(ax):
    items = [
        ('global_runoff_onset_median_composite', 'median runoff onset', 49, 202),
        ('global_runoff_onset_mad_composite', 'median absolute deviation',
         326, 477),
        ('global_temporal_resolution_median_composite',
         'median temporal resolution', 604, 757),
    ]
    for name, label, y0, ylab in items:
        add_img(ax, name, 1836, y0, 536, 273, zorder=5)
        box_label(ax, 2106, ylab, 452, 57, label, fs=27, r=24, lw=3)

## Assemble and save

In [ ]:
fig = plt.figure(figsize=(24.34, 10.53), dpi=100)
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, 2434)
ax.set_ylim(1053, 0)
ax.axis('off')

draw_panels(ax)
draw_snow_phenology(ax)
draw_s1_stacks(ax)
draw_pink_arrow(ax)
draw_backscatter_plot(ax)
draw_left_connections(ax)
draw_tile_products(ax)
draw_annual_stacks(ax)
draw_composites(ax)

fig.savefig(OUT, dpi=100, facecolor='white')
plt.show()
